In [5]:
import os
from glob import glob

covid_images = glob(os.path.join('data/dataset/covid', '*'))
print(covid_images[:5])

['data/dataset/covid\\01E392EE-69F9-4E33-BFCE-E5C968654078.jpeg', 'data/dataset/covid\\1-s2.0-S0140673620303706-fx1_lrg.jpg', 'data/dataset/covid\\1-s2.0-S0929664620300449-gr2_lrg-a.jpg', 'data/dataset/covid\\1-s2.0-S0929664620300449-gr2_lrg-b.jpg', 'data/dataset/covid\\1-s2.0-S0929664620300449-gr2_lrg-c.jpg']


In [6]:
normal_images = glob(os.path.join('data/dataset/normal', '*'))
print(normal_images[:5])

['data/dataset/normal\\IM-0115-0001.jpeg', 'data/dataset/normal\\IM-0131-0001.jpeg', 'data/dataset/normal\\IM-0154-0001.jpeg', 'data/dataset/normal\\IM-0225-0001.jpeg', 'data/dataset/normal\\IM-0299-0001.jpeg']


In [7]:
import copy
import random
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

In [8]:

# Reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [9]:

# Config
DATA_DIR = "data/dataset"
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15
VAL_SPLIT = 0.2
LEARNING_RATE = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cpu


In [10]:

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [11]:

# Build datasets from folder structure:
# data/dataset/
#   covid/
#   normal/
base_dataset = datasets.ImageFolder(DATA_DIR)
class_names = base_dataset.classes
num_samples = len(base_dataset)
indices = list(range(num_samples))
random.shuffle(indices)

val_size = int(num_samples * VAL_SPLIT)
val_indices = indices[:val_size]
train_indices = indices[val_size:]

train_dataset = Subset(datasets.ImageFolder(DATA_DIR, transform=train_transform), train_indices)
val_dataset = Subset(datasets.ImageFolder(DATA_DIR, transform=val_transform), val_indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Classes:", class_names)
print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")


# Model: ResNet18 transfer learning
try:
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
except Exception:
    print("Could not load pretrained weights, training from scratch.")
    model = models.resnet18(weights=None)

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 1)
model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_model_wts = copy.deepcopy(model.state_dict())
best_val_loss = float("inf")
patience = 4
patience_counter = 0

for epoch in range(EPOCHS):
    start = time.time()

    # Train
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.float().unsqueeze(1).to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * images.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        train_correct += (preds == labels).sum().item()
        train_total += labels.numel()

    train_loss = train_loss_sum / len(train_dataset)
    train_acc = train_correct / train_total

    # Validation
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)

            logits = model(images)
            loss = criterion(logits, labels)

            val_loss_sum += loss.item() * images.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total += labels.numel()

    val_loss = val_loss_sum / len(val_dataset)
    val_acc = val_correct / val_total

    elapsed = time.time() - start
    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f} | "
        f"time={elapsed:.1f}s"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

# Load best model and save
model.load_state_dict(best_model_wts)

save_path = "covid_xray_resnet18.pth"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "class_names": class_names,
        "image_size": IMAGE_SIZE,
        "threshold": 0.5,
    },
    save_path,
)

print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Saved model to {save_path}")

Classes: ['covid', 'normal']
Train samples: 76 | Val samples: 18


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\LENOVO/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:04<00:00, 10.9MB/s]


Epoch 1/15 | train_loss=0.5223, train_acc=0.8158 | val_loss=0.3885, val_acc=1.0000 | time=8.5s
Epoch 2/15 | train_loss=0.2136, train_acc=0.9605 | val_loss=0.2188, val_acc=1.0000 | time=7.7s
Epoch 3/15 | train_loss=0.0783, train_acc=1.0000 | val_loss=0.1317, val_acc=1.0000 | time=9.4s
Epoch 4/15 | train_loss=0.0261, train_acc=1.0000 | val_loss=0.0861, val_acc=1.0000 | time=12.7s
Epoch 5/15 | train_loss=0.0195, train_acc=1.0000 | val_loss=0.0469, val_acc=1.0000 | time=11.8s
Epoch 6/15 | train_loss=0.0072, train_acc=1.0000 | val_loss=0.0369, val_acc=1.0000 | time=11.8s
Epoch 7/15 | train_loss=0.0051, train_acc=1.0000 | val_loss=0.0262, val_acc=1.0000 | time=12.8s
Epoch 8/15 | train_loss=0.0036, train_acc=1.0000 | val_loss=0.0214, val_acc=1.0000 | time=13.5s
Epoch 9/15 | train_loss=0.0055, train_acc=1.0000 | val_loss=0.0190, val_acc=1.0000 | time=13.7s
Epoch 10/15 | train_loss=0.0105, train_acc=1.0000 | val_loss=0.0231, val_acc=1.0000 | time=13.4s
Epoch 11/15 | train_loss=0.0028, train_acc